# VOICE ChatBOT

1. 사용자 입력을 음성으로 받는다.
2. STT를 적용하여 텍스트로 변환한다.
3. 변환된 텍스트를 입력으로 하여 프롬프트 엔지니어링을 해 api 요청을 보낸다.
4. 반환받은 응답을 TTS를 적용하여 음성으로 재생한다.
 (+) 2에서 입력된 텍스트와 4에서 반환된 응답을 채팅 내역 보듯이 (카카오톡 대화처럼) 현출되도록 출력한다.

In [ ]:
# !pip install faster-whisper gtts

In [ ]:
# !pip install streamlit

In [ ]:
# !pip install streamlit_mic_recorder

  Using cached streamlit_mic_recorder-0.0.8-py3-none-any.whl.metadata (7.7 kB)
  Using cached streamlit-1.55.0-py3-none-any.whl.metadata (9.8 kB)
  Using cached altair-6.0.0-py3-none-any.whl.metadata (11 kB)
  Using cached blinker-1.9.0-py3-none-any.whl.metadata (1.6 kB)
  Using cached cachetools-7.0.5-py3-none-any.whl.metadata (5.6 kB)
  Using cached gitpython-3.1.46-py3-none-any.whl.metadata (13 kB)
  Using cached pandas-2.3.3-cp312-cp312-win_amd64.whl.metadata (19 kB)
  Using cached pydeck-0.9.1-py2.py3-none-any.whl.metadata (4.1 kB)
  Using cached toml-0.10.2-py2.py3-none-any.whl.metadata (7.1 kB)
  Using cached watchdog-6.0.0-py3-none-win_amd64.whl.metadata (44 kB)
  Using cached narwhals-2.18.0-py3-none-any.whl.metadata (14 kB)
  Using cached gitdb-4.0.12-py3-none-any.whl.metadata (1.2 kB)
  Using cached smmap-5.0.3-py3-none-any.whl.metadata (4.6 kB)
Using cached streamlit_mic_recorder-0.0.8-py3-none-any.whl (2.2 MB)
Using cached streamlit-1.55.0-py3-none-any.whl (9.1 MB)
Using c

In [ ]:
import speech_recognition as sr
from IPython.display import Audio, display
from faster_whisper import WhisperModel
from gtts import gTTS
import os
import time

# 모듈 불러오기
from trpg_gm_module import create_trpg_world, summarize_for_player

# [1] STT 모델 로드
print("음성 인식기(STT) 로딩 중...")
stt_model = WhisperModel("base", device="cuda", compute_type="float16")

# [2] 대화 기록 초기화 (시스템 설정)
history = [
    {
        "role": "system", 
        "content": "당신은 TRPG의 게임 진행자입니다. 플레이어의 선택에 따라 세계관을 확장하고 몰입감 있는 스토리를 이어가세요."
    }
]

def run_voice_trpg_turn():
    global history # 대화 기록 유지를 위해 전역 변수 사용
    
    r = sr.Recognizer()
    with sr.Microphone(sample_rate=16000) as source:
        print("\n [지금 말씀하세요] 진행자가 듣고 있습니다...")
        r.adjust_for_ambient_noise(source, duration=1)
        try:
            audio_data = r.listen(source, timeout=5, phrase_time_limit=10)
        except sr.WaitTimeoutError:
            print("음성이 감지되지 않았습니다.")
            return True # 계속 진행 시도

    # 음성 -> 텍스트
    temp_input = "user_input.wav"
    with open(temp_input, "wb") as f:
        f.write(audio_data.get_wav_data())

    segments, _ = stt_model.transcribe(temp_input, language="ko")
    user_text = "".join([s.text for s in segments])
    
    if not user_text.strip():
        print("음성이 감지되지 않았습니다.")
        return True

    print(f"💬 나: {user_text}")

    # [3] 대화 기록에 플레이어 입력 추가
    history.append({"role": "user", "content": f"플레이어의 선언: {user_text}"})

    # [4] 마스터의 상세 스토리 생성 (기록 전체 전달)
    master_lore = create_trpg_world(history)
    
    # [5] 마스터의 대답을 기록에 추가 (GM의 '기억'이 됨)
    history.append({"role": "assistant", "content": master_lore})

    # [6] 플레이어용 요약문 생성 및 출력
    player_summary = summarize_for_player(master_lore)
    print(f" 마스터: {player_summary}")

    # TTS 및 재생
    output_audio = f"master_voice_{int(time.time())}.mp3"
    tts = gTTS(text=player_summary, lang='ko')
    tts.save(output_audio)
    display(Audio(output_audio, autoplay=True))
    
    # 계속 진행할지 여부 (예: '종료'라고 말하면 멈춤)
    if "종료" in user_text:
        print("🎮 게임을 종료합니다.")
        return False
    return True

# --- 게임 루프 시작 ---
print("🎲 TRPG 세션을 시작합니다! '종료'라고 말씀하시면 멈춥니다.")
is_running = True
while is_running:
    is_running = run_voice_trpg_turn()
    # 다음 턴 사이에 약간의 간격
    time.sleep(2)

c:\Users\kwonm\anaconda3\envs\nlp_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ [TRPG GM 모듈] 준비 완료!
👂 음성 인식기(STT) 로딩 중...

🎤 [지금 말씀하세요] 마스터가 듣고 있습니다... (약 5초)
✅ 녹음 완료! 분석 중...
💬 나:  현대 시대인데 마법과 과학이 약간 섞여 있었으면 좋겠지
🧙 마스터: 빛나는 네온사인 사이로 마법진이 반짝이는 이 도시, 과학과 마법이 융합된 세계에 오신 것을 환영합니다. 당신은 비밀 연구소와 마법사 길드 사이의 미묘한 권력 다툼 속에서 균형을 지키거나 무너뜨릴 운명을 지녔습니다. 이제, 당신은 이 도시의 미래를 위해 어떤 선택을 하시겠습니까?
